In [3]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 47.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import spacy
import random
from spacy.training import Example
from spacy.pipeline.textcat_multilabel import DEFAULT_MULTI_TEXTCAT_MODEL

# We read only the first 4000 rows
reviews_df = pd.read_csv('/Reviews.csv', nrows=4000)
reviews_df = reviews_df[['Text', 'Score']].dropna()

# Re-labeling the scores for binary classification
# 1, 2, 3 stars as negative (0) and 4, 5 stars as positive (1)
reviews_df.loc[reviews_df.Score <= 3, 'Score'] = 0
reviews_df.loc[reviews_df.Score >= 4, 'Score'] = 1

# Creating pipeline and TextCategorizer component
nlp = spacy.load("en_core_web_md")
config = {
    "threshold": 0.5,
    "model": DEFAULT_MULTI_TEXTCAT_MODEL
}
textcat = nlp.add_pipe("textcat_multilabel", config=config)

# Formating training data
train_examples = []
for index, row in reviews_df.iterrows():
    text = row["Text"]
    rating = row["Score"]
    label = {"POS": True, "NEG": False} if rating == 1 else {"NEG": True, "POS": False}
    train_examples.append(Example.from_dict(nlp.make_doc(text), {"cats": label}))

# Adding labels and initializing
textcat.add_label("POS")
textcat.add_label("NEG")
textcat.initialize(lambda: train_examples, nlp=nlp)

# Training loop
epochs = 10 # Using more epochs than asked
with nlp.select_pipes(enable="textcat_multilabel"):
    optimizer = nlp.resume_training()
    for i in range(epochs):
        random.shuffle(train_examples)
        for example in train_examples:
            nlp.update([example], sgd=optimizer)

print("Training complete!")

# Testing the text classifier
doc2 = nlp("This is the best food I ever ate")
print("\nPrediction for 'This is the best food I ever ate':")
print(doc2.cats)

doc3 = nlp("This food is so bad")
print("\nPrediction for 'This food is so bad':")
print(doc3.cats)

Training complete!

Prediction for 'This is the best food I ever ate':
{'POS': 0.990761935710907, 'NEG': 0.009519594721496105}

Prediction for 'This food is so bad':
{'POS': 0.11748594045639038, 'NEG': 0.8904861807823181}
